**Downloading the data**

In [1]:
!pip install -q kagglehub
import kagglehub
path = kagglehub.dataset_download("kazanova/sentiment140")
print(path)

100%|██████████| 80.9M/80.9M [00:04<00:00, 18.9MB/s]

Extracting files...


/root/.cache/kagglehub/datasets/kazanova/sentiment140/versions/2


### 1. Load the dataset and display number of samples for each class label

In [36]:
import os, string
import numpy as np
import pandas as pd

cols = ["target", "ids", "date", "flag", "user", "text"]
df = pd.read_csv(os.path.join(path, "training.1600000.processed.noemoticon.csv"),
                 encoding="latin-1", names=cols)
df = df[["target", "text"]]

# Number of samples per class label (0 = neg, 1 = neutral, 4 = pos)
label_counts = df["target"].value_counts().reindex([0, 1, 4], fill_value=0)
label_counts.index = ["0 (negative)", "1 (neutral)", "4 (positive)"]
print(label_counts)
print("Note: the dataset has NO neutral tweets (0 samples), so the task is binary (neg vs pos).")

0 (negative)    800000
1 (neutral)          0
4 (positive)    800000
Name: count, dtype: int64
Note: the dataset has NO neutral tweets (0 samples), so the task is binary (neg vs pos).


In [37]:
# NOTE: the full dataset (1.6M tweets) is too slow for 4 LSTM models, so a random sample of 100,000 tweets is used.
SAMPLE_SIZE = 100000
if SAMPLE_SIZE is not None and len(df) > SAMPLE_SIZE:
    df = df.sample(SAMPLE_SIZE, random_state=1337)
df["label"] = df["target"].replace({4: 1})   # 0 = neg, 4 = pos -> 0 / 1

### 2. Clean it to remove stop-words and punctuation

In [4]:
import nltk
nltk.download("stopwords")
from nltk.corpus import stopwords

stop_words = set(stopwords.words("english"))
punct_table = str.maketrans("", "", string.punctuation)

def clean_text(text):
    text = text.lower().translate(punct_table)
    return " ".join(w for w in text.split() if w not in stop_words)

df["clean_text"] = df["text"].apply(clean_text)
df = df[df["clean_text"].str.len() > 0]
df[["text", "clean_text"]].head()

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


,text,clean_text
848693,@TraceyHewins thank you.hope you are gettin mo...,traceyhewins thank youhope gettin followers we...
258778,@thingsofthesky I feel all the way round ever...,thingsofthesky feel way round everythings chan...
931969,@authalic Because they write amazing music &am...,authalic write amazing music amp theyre great ...
849628,wishes we all could hang out again. Stop actin...,wishes could hang stop acting weird around hug...
163699,Wish I was @ wembley,wish wembley


### 3. Split dataset in train and test with 70-30 ratio (stratified), validation = 20% of the training data

In [5]:
import tensorflow as tf
from tensorflow import keras
from sklearn.model_selection import train_test_split

X_train_full, X_test, y_train_full, y_test = train_test_split(
    df["clean_text"].values, df["label"].values.astype("float32"),
    test_size=0.3, stratify=df["label"].values, random_state=1337)

X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full,
    test_size=0.2, stratify=y_train_full, random_state=1337)

print(f"Train: {len(X_train)}, Val: {len(X_val)}, Test: {len(X_test)}")

batch_size = 128
train_ds = tf.data.Dataset.from_tensor_slices((X_train, y_train)).batch(batch_size)
val_ds = tf.data.Dataset.from_tensor_slices((X_val, y_val)).batch(batch_size)
test_ds = tf.data.Dataset.from_tensor_slices((X_test, y_test)).batch(batch_size)
text_only_train_ds = train_ds.map(lambda x, y: x)

Train: 55992, Val: 13999, Test: 29997


**Our model-building utility (two hidden layers with dim 32 and 16)**

In [6]:
from tensorflow import keras
from tensorflow.keras import layers

def get_model(max_tokens=10000, hidden_dim=32, hidden_dim2=16):
    inputs = keras.Input(shape=(max_tokens,))
    x = layers.Dense(hidden_dim, activation="relu")(inputs)
    x = layers.Dense(hidden_dim2, activation="relu")(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(1, activation="sigmoid")(x)
    model = keras.Model(inputs, outputs)
    model.compile(optimizer="rmsprop",
                  loss="binary_crossentropy",
                  metrics=["accuracy"])
    return model

**Our evaluation utility (accuracy, precision, recall and F1 on the test dataset)**

In [7]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

results = {}

def evaluate_model(name, model, test_ds, y_true):
    y_pred = (model.predict(test_ds) > 0.5).astype(int).ravel()
    row = {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred),
        "Recall": recall_score(y_true, y_pred),
        "F1": f1_score(y_true, y_pred),
    }
    results[name] = row
    print(f"Test acc: {row['Accuracy']:.3f}")
    print(pd.DataFrame(row, index=[name]).round(4))

### Experiment 1: TextVectorization, one-gram, multi_hot (max_tokens = 10000)

In [8]:
from tensorflow.keras.layers import TextVectorization

text_vectorization = TextVectorization(
    max_tokens=10000,
    output_mode="multi_hot",
)
text_vectorization.adapt(text_only_train_ds)

binary_1gram_train_ds = train_ds.map(
    lambda x, y: (text_vectorization(x), y),
    num_parallel_calls=4)
binary_1gram_val_ds = val_ds.map(
    lambda x, y: (text_vectorization(x), y),
    num_parallel_calls=4)
binary_1gram_test_ds = test_ds.map(
    lambda x, y: (text_vectorization(x), y),
    num_parallel_calls=4)

In [9]:
model = get_model(max_tokens=len(text_vectorization.get_vocabulary()))
model.summary()
callbacks = [
    keras.callbacks.ModelCheckpoint("binary_1gram.keras",
                                    save_best_only=True)
]
model.fit(binary_1gram_train_ds,
          validation_data=binary_1gram_val_ds,
          epochs=5,
          callbacks=callbacks)
model = keras.models.load_model("binary_1gram.keras")
evaluate_model("1. 1-gram multi_hot", model, binary_1gram_test_ds, y_test)

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 10000)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 32)             │       320,032 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 16)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 320,577 (1.22 MB)

 Trainable params: 320,577 (1.22 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/5
438/438 ━━━━━━━━━━━━━━━━━━━━ 9s 18ms/step - accuracy: 0.7144 - loss: 0.5757 - val_accuracy: 0.7555 - val_loss: 0.5116
Epoch 2/5
438/438 ━━━━━━━━━━━━━━━━━━━━ 7s 16ms/step - accuracy: 0.7760 - loss: 0.5060 - val_accuracy: 0.7613 - val_loss: 0.5050
Epoch 3/5
438/438 ━━━━━━━━━━━━━━━━━━━━ 18s 41ms/step - accuracy: 0.7915 - loss: 0.4841 - val_accuracy: 0.7606 - val_loss: 0.5077
Epoch 4/5
438/438 ━━━━━━━━━━━━━━━━━━━━ 17s 39ms/step - accuracy: 0.8021 - loss: 0.4655 - val_accuracy: 0.7618 - val_loss: 0.5113
Epoch 5/5
438/438 ━━━━━━━━━━━━━━━━━━━━ 17s 38ms/step - accuracy: 0.8114 - loss: 0.4503 - val_accuracy: 0.7587 - val_loss: 0.5145
235/235 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step
Test acc: 0.763
                     Accuracy  Precision  Recall      F1
1. 1-gram multi_hot    0.7628     0.7474  0.7911  0.7686


### Experiment 2: TextVectorization, two-gram, multi_hot (max_tokens = 10000)

In [10]:
text_vectorization = TextVectorization(
    ngrams=2,
    max_tokens=10000,
    output_mode="multi_hot",
)
text_vectorization.adapt(text_only_train_ds)

binary_2gram_train_ds = train_ds.map(
    lambda x, y: (text_vectorization(x), y),
    num_parallel_calls=4)
binary_2gram_val_ds = val_ds.map(
    lambda x, y: (text_vectorization(x), y),
    num_parallel_calls=4)
binary_2gram_test_ds = test_ds.map(
    lambda x, y: (text_vectorization(x), y),
    num_parallel_calls=4)

model = get_model(max_tokens=len(text_vectorization.get_vocabulary()))
model.summary()
callbacks = [
    keras.callbacks.ModelCheckpoint("binary_2gram.keras",
                                    save_best_only=True)
]
model.fit(binary_2gram_train_ds,
          validation_data=binary_2gram_val_ds,
          epochs=5,
          callbacks=callbacks)
model = keras.models.load_model("binary_2gram.keras")
evaluate_model("2. 2-gram multi_hot", model, binary_2gram_test_ds, y_test)

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 10000)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 32)             │       320,032 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 16)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 320,577 (1.22 MB)

 Trainable params: 320,577 (1.22 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/5
438/438 ━━━━━━━━━━━━━━━━━━━━ 8s 17ms/step - accuracy: 0.7140 - loss: 0.5709 - val_accuracy: 0.7553 - val_loss: 0.5040
Epoch 2/5
438/438 ━━━━━━━━━━━━━━━━━━━━ 14s 26ms/step - accuracy: 0.7785 - loss: 0.4926 - val_accuracy: 0.7602 - val_loss: 0.5002
Epoch 3/5
438/438 ━━━━━━━━━━━━━━━━━━━━ 17s 19ms/step - accuracy: 0.7948 - loss: 0.4691 - val_accuracy: 0.7602 - val_loss: 0.5016
Epoch 4/5
438/438 ━━━━━━━━━━━━━━━━━━━━ 8s 19ms/step - accuracy: 0.8050 - loss: 0.4524 - val_accuracy: 0.7615 - val_loss: 0.5072
Epoch 5/5
438/438 ━━━━━━━━━━━━━━━━━━━━ 9s 19ms/step - accuracy: 0.8127 - loss: 0.4399 - val_accuracy: 0.7595 - val_loss: 0.5105
235/235 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step
Test acc: 0.766
                     Accuracy  Precision  Recall      F1
2. 2-gram multi_hot    0.7657     0.7499  0.7945  0.7715


### Experiment 3: TextVectorization, two-gram, tf_idf (max_tokens = 10000)

In [11]:
text_vectorization = TextVectorization(
    ngrams=2,
    max_tokens=10000,
    output_mode="tf_idf",
)
text_vectorization.adapt(text_only_train_ds)

tfidf_2gram_train_ds = train_ds.map(
    lambda x, y: (text_vectorization(x), y),
    num_parallel_calls=4)
tfidf_2gram_val_ds = val_ds.map(
    lambda x, y: (text_vectorization(x), y),
    num_parallel_calls=4)
tfidf_2gram_test_ds = test_ds.map(
    lambda x, y: (text_vectorization(x), y),
    num_parallel_calls=4)

model = get_model(max_tokens=len(text_vectorization.get_vocabulary()))
model.summary()
callbacks = [
    keras.callbacks.ModelCheckpoint("tfidf_2gram.keras",
                                    save_best_only=True)
]
model.fit(tfidf_2gram_train_ds,
          validation_data=tfidf_2gram_val_ds,
          epochs=5,
          callbacks=callbacks)
model = keras.models.load_model("tfidf_2gram.keras")
evaluate_model("3. 2-gram tf_idf", model, tfidf_2gram_test_ds, y_test)

Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_2 (InputLayer)      │ (None, 10000)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 32)             │       320,032 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 16)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 320,577 (1.22 MB)

 Trainable params: 320,577 (1.22 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/5
438/438 ━━━━━━━━━━━━━━━━━━━━ 10s 20ms/step - accuracy: 0.7040 - loss: 0.5766 - val_accuracy: 0.7564 - val_loss: 0.5022
Epoch 2/5
438/438 ━━━━━━━━━━━━━━━━━━━━ 8s 19ms/step - accuracy: 0.7739 - loss: 0.4918 - val_accuracy: 0.7615 - val_loss: 0.4997
Epoch 3/5
438/438 ━━━━━━━━━━━━━━━━━━━━ 7s 17ms/step - accuracy: 0.7907 - loss: 0.4630 - val_accuracy: 0.7599 - val_loss: 0.5085
Epoch 4/5
438/438 ━━━━━━━━━━━━━━━━━━━━ 11s 19ms/step - accuracy: 0.8038 - loss: 0.4455 - val_accuracy: 0.7607 - val_loss: 0.5071
Epoch 5/5
438/438 ━━━━━━━━━━━━━━━━━━━━ 10s 18ms/step - accuracy: 0.8126 - loss: 0.4284 - val_accuracy: 0.7617 - val_loss: 0.5154
235/235 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step
Test acc: 0.764
                  Accuracy  Precision  Recall      F1
3. 2-gram tf_idf     0.764     0.7637  0.7617  0.7627


**Preparing integer sequence datasets**

In [12]:
max_length = 140
max_tokens = 10000
text_vectorization = TextVectorization(
    max_tokens=max_tokens,
    output_mode="int",
    output_sequence_length=max_length,
)
text_vectorization.adapt(text_only_train_ds)

int_train_ds = train_ds.map(
    lambda x, y: (text_vectorization(x), y),
    num_parallel_calls=4)
int_val_ds = val_ds.map(
    lambda x, y: (text_vectorization(x), y),
    num_parallel_calls=4)
int_test_ds = test_ds.map(
    lambda x, y: (text_vectorization(x), y),
    num_parallel_calls=4)

### Experiment 4: Model that uses an `Embedding` layer trained from scratch (LSTM, output dim = 32)

In [13]:
inputs = keras.Input(shape=(None,), dtype="int64")
embedded = layers.Embedding(
    input_dim=max_tokens, output_dim=256, mask_zero=True)(inputs)
x = layers.LSTM(32)(embedded)
x = layers.Dropout(0.5)(x)
outputs = layers.Dense(1, activation="sigmoid")(x)
model = keras.Model(inputs, outputs)
model.compile(optimizer="rmsprop",
              loss="binary_crossentropy",
              metrics=["accuracy"])
model.summary()

callbacks = [
    keras.callbacks.ModelCheckpoint("embeddings_lstm.keras",
                                    save_best_only=True)
]
model.fit(int_train_ds, validation_data=int_val_ds, epochs=5, callbacks=callbacks)
model = keras.models.load_model("embeddings_lstm.keras")
evaluate_model("4. Embedding layer + LSTM", model, int_test_ds, y_test)

Model: "functional_3"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_3       │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding           │ (None, None, 256) │  2,560,000 │ input_layer_3[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal           │ (None, None)      │          0 │ input_layer_3[0]… │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm (LSTM)         │ (None, 32)        │     36,992 │ embedding[0][0],  │
│                     │                   │            │ not_equal[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_3 (Dropout) │ (None, 32)        │          0 │ lstm[0][0]        │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_9 (Dense)     │ (None, 1)         │         33 │ dropout_3[0][0]   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 2,597,025 (9.91 MB)

 Trainable params: 2,597,025 (9.91 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/5
438/438 ━━━━━━━━━━━━━━━━━━━━ 165s 371ms/step - accuracy: 0.7190 - loss: 0.5594 - val_accuracy: 0.7534 - val_loss: 0.5036
Epoch 2/5
438/438 ━━━━━━━━━━━━━━━━━━━━ 147s 335ms/step - accuracy: 0.7748 - loss: 0.4854 - val_accuracy: 0.7603 - val_loss: 0.5013
Epoch 3/5
438/438 ━━━━━━━━━━━━━━━━━━━━ 205s 342ms/step - accuracy: 0.7925 - loss: 0.4606 - val_accuracy: 0.7643 - val_loss: 0.5047
Epoch 4/5
438/438 ━━━━━━━━━━━━━━━━━━━━ 145s 332ms/step - accuracy: 0.8045 - loss: 0.4432 - val_accuracy: 0.7626 - val_loss: 0.5106
Epoch 5/5
438/438 ━━━━━━━━━━━━━━━━━━━━ 143s 327ms/step - accuracy: 0.8148 - loss: 0.4268 - val_accuracy: 0.7598 - val_loss: 0.5206
235/235 ━━━━━━━━━━━━━━━━━━━━ 17s 71ms/step
Test acc: 0.761
                           Accuracy  Precision  Recall      F1
4. Embedding layer + LSTM    0.7605     0.7389  0.8027  0.7695


### Experiment 5: Using pretrained word embeddings (GloVe)

In [14]:
!wget https://downloads.cs.stanford.edu/nlp/data/glove.6B.zip
!unzip -q glove.6B.zip glove.6B.100d.txt

--2026-10-03 07:27:39--  https://downloads.cs.stanford.edu/nlp/data/glove.6B.zip
Resolving downloads.cs.stanford.edu (downloads.cs.stanford.edu)... 171.64.64.22
Connecting to downloads.cs.stanford.edu (downloads.cs.stanford.edu)|171.64.64.22|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 862182613 (822M) [application/zip]
Saving to: ‘glove.6B.zip’

glove.6B.zip        100%[===================>] 822.24M  4.87MB/s    in 2m 55s  

2026-10-03 07:30:36 (4.69 MB/s) - ‘glove.6B.zip’ saved [862182613/862182613]



**Parsing the GloVe word-embeddings file**

In [28]:
path_to_glove_file = "glove.6B.100d.txt"

embeddings_index = {}
with open(path_to_glove_file, encoding="utf8") as f:
    for line in f:
        word, coefs = line.split(maxsplit=1)
        coefs = np.asarray(coefs.split(), dtype="float32")
        embeddings_index[word] = coefs

print(f"Found {len(embeddings_index)} word vectors.")

Found 400000 word vectors.


**Preparing the GloVe word-embeddings matrix**

In [16]:
embedding_dim = 100

vocabulary = text_vectorization.get_vocabulary()
word_index = dict(zip(vocabulary, range(len(vocabulary))))

embedding_matrix = np.zeros((max_tokens, embedding_dim))
for word, i in word_index.items():
    if i < max_tokens:
        embedding_vector = embeddings_index.get(word)
        if embedding_vector is not None:
            embedding_matrix[i] = embedding_vector

In [17]:
embedding_layer = layers.Embedding(
    max_tokens,
    embedding_dim,
    embeddings_initializer=keras.initializers.Constant(embedding_matrix),
    trainable=False,
    mask_zero=True,
)

**Model that uses a pretrained Embedding layer**

In [18]:
inputs = keras.Input(shape=(None,), dtype="int64")
embedded = embedding_layer(inputs)
x = layers.LSTM(32)(embedded)
x = layers.Dropout(0.5)(x)
outputs = layers.Dense(1, activation="sigmoid")(x)
model = keras.Model(inputs, outputs)
model.compile(optimizer="rmsprop",
              loss="binary_crossentropy",
              metrics=["accuracy"])
model.summary()

callbacks = [
    keras.callbacks.ModelCheckpoint("glove_embeddings_sequence_model.keras",
                                    save_best_only=True)
]
model.fit(int_train_ds, validation_data=int_val_ds, epochs=5, callbacks=callbacks)
model = keras.models.load_model("glove_embeddings_sequence_model.keras")
evaluate_model("5. GloVe + LSTM", model, int_test_ds, y_test)

Model: "functional_4"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_4       │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_1         │ (None, None, 100) │  1,000,000 │ input_layer_4[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal_2         │ (None, None)      │          0 │ input_layer_4[0]… │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_1 (LSTM)       │ (None, 32)        │     17,024 │ embedding_1[0][0… │
│                     │                   │            │ not_equal_2[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_4 (Dropout) │ (None, 32)        │          0 │ lstm_1[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_10 (Dense)    │ (None, 1)         │         33 │ dropout_4[0][0]   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 1,017,057 (3.88 MB)

 Trainable params: 17,057 (66.63 KB)

 Non-trainable params: 1,000,000 (3.81 MB)

Epoch 1/5
438/438 ━━━━━━━━━━━━━━━━━━━━ 91s 203ms/step - accuracy: 0.6756 - loss: 0.6012 - val_accuracy: 0.7116 - val_loss: 0.5595
Epoch 2/5
438/438 ━━━━━━━━━━━━━━━━━━━━ 127s 170ms/step - accuracy: 0.7150 - loss: 0.5597 - val_accuracy: 0.7274 - val_loss: 0.5392
Epoch 3/5
438/438 ━━━━━━━━━━━━━━━━━━━━ 83s 172ms/step - accuracy: 0.7284 - loss: 0.5432 - val_accuracy: 0.7373 - val_loss: 0.5274
Epoch 4/5
438/438 ━━━━━━━━━━━━━━━━━━━━ 77s 176ms/step - accuracy: 0.7366 - loss: 0.5304 - val_accuracy: 0.7414 - val_loss: 0.5214
Epoch 5/5
438/438 ━━━━━━━━━━━━━━━━━━━━ 78s 179ms/step - accuracy: 0.7421 - loss: 0.5218 - val_accuracy: 0.7433 - val_loss: 0.5145
235/235 ━━━━━━━━━━━━━━━━━━━━ 13s 53ms/step
Test acc: 0.747
                 Accuracy  Precision  Recall      F1
5. GloVe + LSTM    0.7472     0.7271  0.7882  0.7564


### Experiment 6: Using pretrained word embeddings (GloVe-twitter)

In [19]:
!wget https://downloads.cs.stanford.edu/nlp/data/glove.twitter.27B.zip
!unzip -q glove.twitter.27B.zip glove.twitter.27B.100d.txt

--2026-10-03 07:38:48--  https://downloads.cs.stanford.edu/nlp/data/glove.twitter.27B.zip
Resolving downloads.cs.stanford.edu (downloads.cs.stanford.edu)... 171.64.64.22
Connecting to downloads.cs.stanford.edu (downloads.cs.stanford.edu)|171.64.64.22|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1520408563 (1.4G) [application/zip]
Saving to: ‘glove.twitter.27B.zip’

glove.twitter.27B.z 100%[===================>]   1.42G  2.94MB/s    in 8m 55s  

2026-10-03 07:47:43 (2.71 MB/s) - ‘glove.twitter.27B.zip’ saved [1520408563/1520408563]



**Parsing the GloVe-twitter word-embeddings file**

In [20]:
path_to_glove_twitter_file = "glove.twitter.27B.100d.txt"

embeddings_index = {}
with open(path_to_glove_twitter_file, encoding="utf8") as f:
    for line in f:
        word, coefs = line.split(maxsplit=1)
        coefs = np.asarray(coefs.split(), dtype="float32")
        embeddings_index[word] = coefs

print(f"Found {len(embeddings_index)} word vectors.")

Found 1193514 word vectors.


**Preparing the GloVe-twitter word-embeddings matrix**

In [21]:
embedding_dim = 100

vocabulary = text_vectorization.get_vocabulary()
word_index = dict(zip(vocabulary, range(len(vocabulary))))

embedding_matrix = np.zeros((max_tokens, embedding_dim))
for word, i in word_index.items():
    if i < max_tokens:
        embedding_vector = embeddings_index.get(word)
        if embedding_vector is not None:
            embedding_matrix[i] = embedding_vector

embedding_layer = layers.Embedding(
    max_tokens,
    embedding_dim,
    embeddings_initializer=keras.initializers.Constant(embedding_matrix),
    trainable=False,
    mask_zero=True,
)

**Model that uses a pretrained Embedding layer**

In [22]:
inputs = keras.Input(shape=(None,), dtype="int64")
embedded = embedding_layer(inputs)
x = layers.LSTM(32)(embedded)
x = layers.Dropout(0.5)(x)
outputs = layers.Dense(1, activation="sigmoid")(x)
model = keras.Model(inputs, outputs)
model.compile(optimizer="rmsprop",
              loss="binary_crossentropy",
              metrics=["accuracy"])
model.summary()

callbacks = [
    keras.callbacks.ModelCheckpoint("glove_twitter_embeddings_sequence_model.keras",
                                    save_best_only=True)
]
model.fit(int_train_ds, validation_data=int_val_ds, epochs=5, callbacks=callbacks)
model = keras.models.load_model("glove_twitter_embeddings_sequence_model.keras")
evaluate_model("6. GloVe-twitter + LSTM", model, int_test_ds, y_test)

Model: "functional_5"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_5       │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_2         │ (None, None, 100) │  1,000,000 │ input_layer_5[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal_4         │ (None, None)      │          0 │ input_layer_5[0]… │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_2 (LSTM)       │ (None, 32)        │     17,024 │ embedding_2[0][0… │
│                     │                   │            │ not_equal_4[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_5 (Dropout) │ (None, 32)        │          0 │ lstm_2[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_11 (Dense)    │ (None, 1)         │         33 │ dropout_5[0][0]   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 1,017,057 (3.88 MB)

 Trainable params: 17,057 (66.63 KB)

 Non-trainable params: 1,000,000 (3.81 MB)

Epoch 1/5
438/438 ━━━━━━━━━━━━━━━━━━━━ 81s 179ms/step - accuracy: 0.7129 - loss: 0.5617 - val_accuracy: 0.7425 - val_loss: 0.5173
Epoch 2/5
438/438 ━━━━━━━━━━━━━━━━━━━━ 85s 194ms/step - accuracy: 0.7414 - loss: 0.5232 - val_accuracy: 0.7503 - val_loss: 0.5035
Epoch 3/5
438/438 ━━━━━━━━━━━━━━━━━━━━ 81s 185ms/step - accuracy: 0.7514 - loss: 0.5100 - val_accuracy: 0.7580 - val_loss: 0.4946
Epoch 4/5
438/438 ━━━━━━━━━━━━━━━━━━━━ 77s 176ms/step - accuracy: 0.7561 - loss: 0.5010 - val_accuracy: 0.7623 - val_loss: 0.4885
Epoch 5/5
438/438 ━━━━━━━━━━━━━━━━━━━━ 78s 178ms/step - accuracy: 0.7605 - loss: 0.4946 - val_accuracy: 0.7636 - val_loss: 0.4844
235/235 ━━━━━━━━━━━━━━━━━━━━ 12s 49ms/step
Test acc: 0.763
                         Accuracy  Precision  Recall      F1
6. GloVe-twitter + LSTM    0.7626     0.7546  0.7753  0.7648


### Experiment 7: Using pretrained word embeddings (FastText)

In [23]:
!wget https://dl.fbaipublicfiles.com/fasttext/vectors-english/wiki-news-300d-1M.vec.zip
!unzip -q wiki-news-300d-1M.vec.zip

--2026-10-03 07:55:47--  https://dl.fbaipublicfiles.com/fasttext/vectors-english/wiki-news-300d-1M.vec.zip
Resolving dl.fbaipublicfiles.com (dl.fbaipublicfiles.com)... 18.65.14.94, 18.65.14.79, 18.65.14.61, ...
Connecting to dl.fbaipublicfiles.com (dl.fbaipublicfiles.com)|18.65.14.94|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 681808098 (650M) [application/zip]
Saving to: ‘wiki-news-300d-1M.vec.zip’

wiki-news-300d-1M.v 100%[===================>] 650.22M  38.3MB/s    in 11s     

2026-10-03 07:55:58 (59.8 MB/s) - ‘wiki-news-300d-1M.vec.zip’ saved [681808098/681808098]



**Parsing the FastText word-embeddings file**

In [24]:
path_to_fasttext_file = "wiki-news-300d-1M.vec"

embeddings_index = {}
with open(path_to_fasttext_file, encoding="utf8") as f:
    next(f)   # first line is the header (number of words and dimension)
    for line in f:
        word, coefs = line.split(maxsplit=1)
        coefs = np.asarray(coefs.split(), dtype="float32")
        embeddings_index[word] = coefs

print(f"Found {len(embeddings_index)} word vectors.")

Found 999994 word vectors.


**Preparing the FastText word-embeddings matrix**

In [25]:
embedding_dim = 300

vocabulary = text_vectorization.get_vocabulary()
word_index = dict(zip(vocabulary, range(len(vocabulary))))

embedding_matrix = np.zeros((max_tokens, embedding_dim))
for word, i in word_index.items():
    if i < max_tokens:
        embedding_vector = embeddings_index.get(word)
        if embedding_vector is not None:
            embedding_matrix[i] = embedding_vector

embedding_layer = layers.Embedding(
    max_tokens,
    embedding_dim,
    embeddings_initializer=keras.initializers.Constant(embedding_matrix),
    trainable=False,
    mask_zero=True,
)

**Model that uses a pretrained Embedding layer**

In [26]:
inputs = keras.Input(shape=(None,), dtype="int64")
embedded = embedding_layer(inputs)
x = layers.LSTM(32)(embedded)
x = layers.Dropout(0.5)(x)
outputs = layers.Dense(1, activation="sigmoid")(x)
model = keras.Model(inputs, outputs)
model.compile(optimizer="rmsprop",
              loss="binary_crossentropy",
              metrics=["accuracy"])
model.summary()

callbacks = [
    keras.callbacks.ModelCheckpoint("fasttext_embeddings_sequence_model.keras",
                                    save_best_only=True)
]
model.fit(int_train_ds, validation_data=int_val_ds, epochs=5, callbacks=callbacks)
model = keras.models.load_model("fasttext_embeddings_sequence_model.keras")
evaluate_model("7. FastText + LSTM", model, int_test_ds, y_test)

Model: "functional_6"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_6       │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_3         │ (None, None, 300) │  3,000,000 │ input_layer_6[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal_6         │ (None, None)      │          0 │ input_layer_6[0]… │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_3 (LSTM)       │ (None, 32)        │     42,624 │ embedding_3[0][0… │
│                     │                   │            │ not_equal_6[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_6 (Dropout) │ (None, 32)        │          0 │ lstm_3[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_12 (Dense)    │ (None, 1)         │         33 │ dropout_6[0][0]   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 3,042,657 (11.61 MB)

 Trainable params: 42,657 (166.63 KB)

 Non-trainable params: 3,000,000 (11.44 MB)

Epoch 1/5
438/438 ━━━━━━━━━━━━━━━━━━━━ 114s 255ms/step - accuracy: 0.7087 - loss: 0.5759 - val_accuracy: 0.7415 - val_loss: 0.5281
Epoch 2/5
438/438 ━━━━━━━━━━━━━━━━━━━━ 123s 281ms/step - accuracy: 0.7387 - loss: 0.5332 - val_accuracy: 0.7461 - val_loss: 0.5174
Epoch 3/5
438/438 ━━━━━━━━━━━━━━━━━━━━ 120s 274ms/step - accuracy: 0.7462 - loss: 0.5231 - val_accuracy: 0.7501 - val_loss: 0.5097
Epoch 4/5
438/438 ━━━━━━━━━━━━━━━━━━━━ 114s 259ms/step - accuracy: 0.7490 - loss: 0.5146 - val_accuracy: 0.7512 - val_loss: 0.5067
Epoch 5/5
438/438 ━━━━━━━━━━━━━━━━━━━━ 141s 257ms/step - accuracy: 0.7529 - loss: 0.5072 - val_accuracy: 0.7550 - val_loss: 0.4997
235/235 ━━━━━━━━━━━━━━━━━━━━ 22s 94ms/step
Test acc: 0.758
                    Accuracy  Precision  Recall      F1
7. FastText + LSTM    0.7582     0.7503  0.7709  0.7605


### Results on the test dataset (30% of Sentiment140)

In [27]:
results_df = pd.DataFrame(results).T.round(4)
print(results_df)
print("Best model (by F1):", results_df["F1"].idxmax())

                           Accuracy  Precision  Recall      F1
1. 1-gram multi_hot          0.7628     0.7474  0.7911  0.7686
2. 2-gram multi_hot          0.7657     0.7499  0.7945  0.7715
3. 2-gram tf_idf             0.7640     0.7637  0.7617  0.7627
4. Embedding layer + LSTM    0.7605     0.7389  0.8027  0.7695
5. GloVe + LSTM              0.7472     0.7271  0.7882  0.7564
6. GloVe-twitter + LSTM      0.7626     0.7546  0.7753  0.7648
7. FastText + LSTM           0.7582     0.7503  0.7709  0.7605
Best model (by F1): 2. 2-gram multi_hot


**Which model performs best and why?**

| # | Model | Accuracy | Precision | Recall | F1 |
|---|---|---|---|---|---|
| 1 | 1-gram multi_hot | 0.7628 | 0.7474 | 0.7911 | 0.7686 |
| 2 | 2-gram multi_hot | **0.7657** | 0.7499 | 0.7945 | **0.7715** |
| 3 | 2-gram tf_idf | 0.7640 | **0.7637** | 0.7617 | 0.7627 |
| 4 | Embedding layer + LSTM | 0.7605 | 0.7389 | **0.8027** | 0.7695 |
| 5 | GloVe (6B) + LSTM | 0.7472 | 0.7271 | 0.7882 | 0.7564 |
| 6 | GloVe-Twitter + LSTM | 0.7626 | 0.7546 | 0.7753 | 0.7648 |
| 7 | FastText + LSTM | 0.7582 | 0.7503 | 0.7709 | 0.7605 |

**Best model:** 2-gram multi_hot (Experiment 2) has the highest accuracy (76.57%) and F1 (77.15%). The gap to models 1, 3, 4 and 6 is under 0.5%, so they perform almost the same.

**Why:**
- Tweets are short, so sentiment comes mostly from single words and short phrases. Bigrams capture these phrases.
- The LSTM gives no gain on short texts, overfits after epoch 2, and is much slower.
- Pre-trained embeddings were frozen and could not adapt to this data. GloVe (6B) is the worst because it is trained on formal text, while GloVe-Twitter is better because it is trained on tweets.

**Conclusion:** 2-gram multi_hot is the best choice: highest scores, fast and simple. Results are from one run on a 100,000-tweet sample with noisy labels.